# rng-throughput
This notebook measures the throughput of random number generation as implemented by the C++ standard library.

```python
for dist in distributions:
    for gen in generators:
        tic()
        for i in range(n_iters):
            x = dist(gen)

        time_s = toc()
        throughput = sizeof(x) * n_iters / time_s
```

The benchmark can be implemented with the python language and benefits from:
  * string interpolation
  * JIT compilation
  * LLVM passes to detect *dead code elimination*

In [1]:
from peppo.ext import Language, ExtSource
import itertools

## Domain specific code
Users of the PEPPO framework declare the program logic using primitive data and functions.
The user code is ingested with the `ExtSource` class, which is responsible of compiling the code into LLVM IR.

Once an LLVM module is created, the limited functionality (as exposed by the `llvmlite` dependency) can be accessed.
This involves optimization passes and JIT compilation, but in principle custom passes can be added.

In [2]:
DISTRIBUTIONS = [
    'normal_distribution<double>',
    'uniform_real_distribution<double>'
]

GENERATORS = [
    'minstd_rand0',
    'minstd_rand',
    'mt19937',
    'mt19937_64',
    'ranlux24_base',
    'ranlux48_base',
    'ranlux24',
    'ranlux48',
    'knuth_b',
]

In [3]:
def generate_benchmark(entry_point: str,
                     distribution: str,
                     generator: str) -> ExtSource:
    # digraphs `<%` are used to not escape the curly braces
    src = f"""
    #include <chrono>
    #include <random>

    using namespace std;

    double {entry_point}(std::size_t nruns) <%
        {distribution} dist;
        {generator} gen;

        const auto t_start = std::chrono::steady_clock::now();
        for (; nruns; --nruns) <%
            const auto x = dist(gen);
        %>
        const auto t_end = std::chrono::steady_clock::now();

        const std::chrono::duration<double> delta = t_end - t_start;
        return delta.count();
    %>
    """

    return ExtSource(src, Language.CPP)

## Exploration with the PEPPO framework

In [4]:
ENTRY_POINT = 'profile_generation'
benchmarks = {}

for dist in DISTRIBUTIONS:
    for gen in GENERATORS:
        bench_src = generate_benchmark(
            ENTRY_POINT,
            dist,
            gen
        )

        benchmarks[(dist, gen)] = bench_src.compile_to_llvm_ir()

In [6]:
bench = benchmarks[(DISTRIBUTIONS[0], GENERATORS[0])]
print(bench)

; ModuleID = '-'
source_filename = "-"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-pc-linux-gnu"

%"class.std::normal_distribution" = type <{ %"struct.std::normal_distribution<>::param_type", double, i8, [7 x i8] }>
%"struct.std::normal_distribution<>::param_type" = type { double, double }
%"class.std::linear_congruential_engine" = type { i64 }
%"class.std::chrono::time_point" = type { %"class.std::chrono::duration" }
%"class.std::chrono::duration" = type { i64 }
%"class.std::chrono::duration.0" = type { double }
%"struct.std::__detail::_Adaptor" = type { ptr }

$_ZNSt19normal_distributionIdEC2Ev = comdat any

$_ZNSt26linear_congruential_engineImLm16807ELm0ELm2147483647EEC2Ev = comdat any

$_ZNSt19normal_distributionIdEclISt26linear_congruential_engineImLm16807ELm0ELm2147483647EEEEdRT_ = comdat any

$_ZNSt6chronomiINS_3_V212steady_clockENS_8durationIlSt5ratioILl1ELl1000000000EEEES6_EENSt11common_typeIJT0_